In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import polars as pl
import numpy as np

# --- 1. SETUP & DATA LOADING ---
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
split_dir = f"{drive_path}/processed_data/final_foundation/splits"

# Load all three splits
train = pl.read_parquet(f"{split_dir}/train.parquet")
val = pl.read_parquet(f"{split_dir}/val.parquet")
test = pl.read_parquet(f"{split_dir}/test.parquet")

# --- 2. GLOBAL MEAN CALCULATION ---
C = train["Rating"].mean()

# --- 3. HYPERPARAMETER TUNING (Validation Set) ---
# We use the Validation set to find the optimal 'm'
m_options = [10, 50, 100, 200, 500]
best_m = None
best_val_rmse = float('inf')

# Pre-calculate training statistics
train_stats = train.group_by("item_id").agg([
    pl.len().alias("v"),
    pl.col("Rating").mean().alias("R")
])

print("--- Tuning 'm' on Validation Set ---")
for m_candidate in m_options:
    # Build candidate model
    candidate_model = train_stats.with_columns(
        ((pl.col("v") / (pl.col("v") + m_candidate)) * pl.col("R") +
         (m_candidate / (pl.col("v") + m_candidate)) * C).alias("pop_score")
    )

    # Evaluate on Validation Set
    val_results = val.join(candidate_model, on="item_id", how="left").fill_null(C)
    val_rmse = np.sqrt(((val_results["Rating"] - val_results["pop_score"])**2).mean())

    print(f"m={m_candidate:3} | Val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_m = m_candidate

print(f"\nOptimal m selected: {best_m}")

# --- 4. EVALUATION FRAMEWORK FUNCTION ---
def calculate_metrics_polars(df, k=10, threshold=7.0):
    """
    Calculates RMSE, Precision@K, Recall@K, and NDCG@K.
    Expected columns: ['user_id', 'Rating', 'pop_score']
    """
    # 1. Ranking Logic
    sorted_df = df.sort(["user_id", "pop_score"], descending=[False, True])
    ranked_df = sorted_df.with_columns(
        pl.col("pop_score").rank(method="ordinal", descending=True).over("user_id").alias("rank")
    )
    top_k = ranked_df.filter(pl.col("rank") <= k)

    # 2. Precision & Recall Components
    user_metrics = ranked_df.group_by("user_id").agg([
        pl.col("Rating").filter(pl.col("Rating") >= threshold).count().alias("n_rel"),
        pl.col("Rating")
            .filter((pl.col("rank") <= k) & (pl.col("Rating") >= threshold))
            .count()
            .alias("n_rel_and_rec_k")
    ])

    user_metrics = user_metrics.with_columns([
        (pl.col("n_rel_and_rec_k") / k).alias("precision"),
        (pl.when(pl.col("n_rel") > 0)
         .then(pl.col("n_rel_and_rec_k") / pl.col("n_rel"))
         .otherwise(0.0)).alias("recall")
    ])

    # 3. NDCG Components
    top_k_dcg = top_k.with_columns(
        (pl.col("Rating") / np.log2(pl.col("rank") + 1)).alias("dcg_item")
    ).group_by("user_id").agg(pl.col("dcg_item").sum().alias("dcg"))

    ideal_top_k = ranked_df.sort(["user_id", "Rating"], descending=[False, True]) \
        .with_columns(pl.col("Rating").rank(method="ordinal", descending=True).over("user_id").alias("ideal_rank")) \
        .filter(pl.col("ideal_rank") <= k)

    top_k_idcg = ideal_top_k.with_columns(
        (pl.col("Rating") / np.log2(pl.col("ideal_rank") + 1)).alias("idcg_item")
    ).group_by("user_id").agg(pl.col("idcg_item").sum().alias("idcg"))

    ndcg_df = top_k_dcg.join(top_k_idcg, on="user_id").with_columns(
        (pl.when(pl.col("idcg") > 0)
         .then(pl.col("dcg") / pl.col("idcg"))
         .otherwise(0.0)).alias("ndcg")
    )

    # 4. Final Aggregation
    return {
        "RMSE": np.sqrt(((df["Rating"] - df["pop_score"])**2).mean()),
        "Precision@10": user_metrics["precision"].mean(),
        "Recall@10": user_metrics["recall"].mean(),
        "NDCG@10": ndcg_df["ndcg"].mean()
    }

# --- 5. FINAL EVALUATION (Test Set) ---
# Re-generate the model using the BEST m found in the tuning step
final_popularity_model = train_stats.with_columns(
    ((pl.col("v") / (pl.col("v") + best_m)) * pl.col("R") +
     (best_m / (pl.col("v") + best_m)) * C).alias("pop_score")
)

test_results = test.join(final_popularity_model, on="item_id", how="left").fill_null(C)
metrics = calculate_metrics_polars(test_results, k=10, threshold=7.0)

print(f"\n--- Final Popularity Baseline Results (Test Set, m={best_m}) ---")
for m_name, m_val in metrics.items():
    print(f"{m_name:12}: {m_val:.4f}")

--- Tuning 'm' on Validation Set ---
m= 10 | Val RMSE: 1.3214
m= 50 | Val RMSE: 1.3279
m=100 | Val RMSE: 1.3350
m=200 | Val RMSE: 1.3453
m=500 | Val RMSE: 1.3632

Optimal m selected: 10

--- Final Popularity Baseline Results (Test Set, m=10) ---
RMSE        : 1.3781
Precision@10: 0.2672
Recall@10   : 0.9060
NDCG@10     : 0.9848
